# 03_main.ipynb — Single-Stage Dual-GPU Training (CIFAR-10 on GPU 0, CIFAR-100 on GPU 1)
This notebook trains 4 configurations per GPU concurrently:
- **GPU 0**: CIFAR-10 (SGD lr 0.10, SGD lr 0.15, Adam lr 0.001, Adam lr 0.002)
- **GPU 1**: CIFAR-100 (SGD lr 0.10, SGD lr 0.15, Adam lr 0.001, Adam lr 0.002)
- Mode: **dev** (45,000 train / 5,000 validation)
- Validation every epoch measuring both RAW and EMA metrics
- Checkpointing and full resumption support with deterministic run IDs
- Time Guard calibration and safe epoch projection


In [ ]:
!nvidia-smi
import torch

print(f"PyTorch Version: {torch.__version__}, Available GPUs: {torch.cuda.device_count()}")


In [ ]:
import os
import shutil

REPO_SOURCE = "git"  # Options: "git" or "kaggle"
REPO_URL = "https://github.com/PhoenixEvo/flower-lite.git"

if REPO_SOURCE == "git":
    if not os.path.exists("flower-lite"):
        os.system(f"git clone {REPO_URL} flower-lite")
    os.chdir("flower-lite")
else:
    if os.path.exists("/kaggle/input/flowerlite-repo"):
        os.system("cp -r /kaggle/input/flowerlite-repo /kaggle/working/flower-lite")
    os.chdir("/kaggle/working/flower-lite")

os.system("pip install -e .")
import flowerlite

print("FlowerLite package successfully imported!")

DATA_ROOT = "/kaggle/working/data"
import torchvision

torchvision.datasets.CIFAR10(root=DATA_ROOT, download=True)
torchvision.datasets.CIFAR100(root=DATA_ROOT, download=True)
print("CIFAR-10 and CIFAR-100 verified.")


In [ ]:
import math
import os
import subprocess
import time

EPOCHS = None  # Human: Set this to an integer (e.g., 60, 80, 100) or leave as None to see safe recommendation
BUDGET_HOURS = 8.0  # Maximum allowed budget for the entire dual-GPU run

def measure_sec_per_epoch(gpu_id="0", batch_size=256, data_root="/kaggle/working/data"):
    print(f"Measuring real execution timing on GPU {gpu_id}...")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu_id}
    probe_id = f"timing_probe_gpu{gpu_id}_{int(time.time())}"
    cmd = [
        "python", "train.py",
        "--config", "configs/base.yaml",
        "--dataset", "cifar10",
        "--optimizer", "sgd",
        "--lr", "0.1",
        "--epochs", "1",
        "--run-id", probe_id,
        "--max-steps", "115",
        "--data-root", data_root,
        "--mode", "dev"
    ]
    
    res = subprocess.run(cmd, env=env, capture_output=True, text=True)
    
    elapsed_100_steps = None
    val_time = None
    for line in res.stdout.split("\n"):
        if "Timing (Steps 10-110)" in line:
            elapsed_100_steps = float(line.split(":")[1].replace("s", "").replace("-", "").strip())
        if "Val Time" in line:
            val_time = float(line.split(":")[1].replace("s", "").replace("-", "").strip())
            
    if elapsed_100_steps is None or val_time is None:
        print("Probe Output STDOUT:\n", res.stdout)
        print("Probe Output STDERR:\n", res.stderr)
        raise RuntimeError("Failed to extract 100-step timing and validation timing from probe output.")
        
    steps_per_epoch = math.ceil(45000 / batch_size) # 176 steps per epoch
    train_time_per_epoch = (elapsed_100_steps / 100.0) * steps_per_epoch
    sec_per_epoch = train_time_per_epoch + val_time
    steps_per_sec = 100.0 / elapsed_100_steps
    
    print(f"--> Measured steps/s: {steps_per_sec:.2f}")
    print(f"--> Train time per epoch (176 steps): {train_time_per_epoch:.2f}s")
    print(f"--> Validation time per epoch (5k images): {val_time:.2f}s")
    print(f"--> Total seconds/epoch: {sec_per_epoch:.2f}s")
    return sec_per_epoch

# Measure timing on GPU 0
sec_per_epoch = measure_sec_per_epoch("0", data_root=DATA_ROOT)

# Calculate safe N (largest N fitting BUDGET_HOURS for 4 configs per GPU)
seconds_per_config_epoch = 4.0 * sec_per_epoch
max_safe_N = math.floor((BUDGET_HOURS * 3600.0) / seconds_per_config_epoch)

print(f"\n================ TIME GUARD SUMMARY ================")
print(f"Measured sec/epoch: {sec_per_epoch:.2f}s")
print(f"Budget: {BUDGET_HOURS} hours")
print(f"Largest safe N (epochs) that fits budget for 4 runs: {max_safe_N}")
print(f"====================================================\n")

if EPOCHS is None:
    raise ValueError(
        f"EPOCHS is currently None! Based on measured {sec_per_epoch:.2f} s/epoch, "
        f"the largest safe epoch count fitting BUDGET_HOURS ({BUDGET_HOURS}h) is {max_safe_N}. "
        f"Please set EPOCHS = {max_safe_N} (or any integer <= {max_safe_N}) in this cell and re-run."
    )

projected_hours = (seconds_per_config_epoch * EPOCHS) / 3600.0
print(f"Configured EPOCHS: {EPOCHS}")
print(f"Projected wall-clock time: {projected_hours:.2f} hours (Budget: {BUDGET_HOURS} hours)")

if projected_hours > BUDGET_HOURS:
    raise RuntimeError(
        f"Projected runtime ({projected_hours:.2f}h) EXCEEDS budget ({BUDGET_HOURS}h)! "
        f"The maximum safe epoch count is {max_safe_N}. Please reduce EPOCHS <= {max_safe_N}."
    )
print("Time guard passed successfully! Proceeding to execution.")


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

working_dir = Path("/kaggle/working")
runs_dir = working_dir / "runs"
runs_dir.mkdir(parents=True, exist_ok=True)
results_csv = working_dir / "results.csv"

# Configuration sequence (4 configs per GPU)
configs_gpu0 = [
    {"dataset": "cifar10", "optimizer": "sgd", "lr": 0.10, "run_id": "c10_sgd_0.10"},
    {"dataset": "cifar10", "optimizer": "sgd", "lr": 0.15, "run_id": "c10_sgd_0.15"},
    {"dataset": "cifar10", "optimizer": "adam", "lr": 0.001, "run_id": "c10_adam_0.001"},
    {"dataset": "cifar10", "optimizer": "adam", "lr": 0.002, "run_id": "c10_adam_0.002"},
]

configs_gpu1 = [
    {"dataset": "cifar100", "optimizer": "sgd", "lr": 0.10, "run_id": "c100_sgd_0.10"},
    {"dataset": "cifar100", "optimizer": "sgd", "lr": 0.15, "run_id": "c100_sgd_0.15"},
    {"dataset": "cifar100", "optimizer": "adam", "lr": 0.001, "run_id": "c100_adam_0.001"},
    {"dataset": "cifar100", "optimizer": "adam", "lr": 0.002, "run_id": "c100_adam_0.002"},
]

# Create worker runner script
runner_script = Path("/kaggle/working/worker_runner.py")
runner_code = '''
import sys
import os
import json
import subprocess
from pathlib import Path

gpu_id = sys.argv[1]
configs_json = sys.argv[2]
epochs = int(sys.argv[3])
data_root = sys.argv[4]

configs = json.loads(configs_json)
log_file = Path(f"/kaggle/working/log_gpu{gpu_id}.txt")

with open(log_file, "a") as f:
    f.write(f"=== Starting GPU {gpu_id} Worker ===\n")

for cfg in configs:
    run_id = cfg["run_id"]
    dataset = cfg["dataset"]
    opt = cfg["optimizer"]
    lr = cfg["lr"]
    
    marker_local = Path("experiments") / run_id / "FINISHED"
    marker_work = Path("/kaggle/working/runs") / run_id / "FINISHED"
    if marker_local.exists() or marker_work.exists():
        with open(log_file, "a") as f:
            f.write(f"[GPU {gpu_id}] Run {run_id} is already FINISHED. Skipping.\n")
        continue

    cmd = [
        sys.executable, "train.py",
        "--config", "configs/base.yaml",
        "--dataset", dataset,
        "--optimizer", opt,
        "--lr", str(lr),
        "--epochs", str(epochs),
        "--run-id", run_id,
        "--data-root", data_root,
        "--mode", "dev",
        "--resume",
        "--results-csv", "/kaggle/working/results.csv",
        "--export-dir", f"/kaggle/working/runs/{run_id}"
    ]

    with open(log_file, "a") as f:
        f.write(f"[GPU {gpu_id}] Launching: {' '.join(cmd)}\n")
        f.flush()
        ret = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
        if ret.returncode != 0:
            f.write(f"[GPU {gpu_id}] Run {run_id} FAILED with return code {ret.returncode}\n")
            sys.exit(ret.returncode)
        else:
            f.write(f"[GPU {gpu_id}] Run {run_id} COMPLETED successfully.\n")

with open(log_file, "a") as f:
    f.write(f"=== GPU {gpu_id} All Configs Done ===\n")
'''
runner_script.write_text(runner_code)

log_gpu0 = "/kaggle/working/log_gpu0.txt"
log_gpu1 = "/kaggle/working/log_gpu1.txt"
if os.path.exists(log_gpu0): os.remove(log_gpu0)
if os.path.exists(log_gpu1): os.remove(log_gpu1)

# Launch workers concurrently
env0 = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
env1 = {**os.environ, "CUDA_VISIBLE_DEVICES": "1"}

p0 = subprocess.Popen([sys.executable, str(runner_script), "0", json.dumps(configs_gpu0), str(EPOCHS), DATA_ROOT], env=env0)
p1 = subprocess.Popen([sys.executable, str(runner_script), "1", json.dumps(configs_gpu1), str(EPOCHS), DATA_ROOT], env=env1)

def print_tail(file_path, num_lines=15):
    if not os.path.exists(file_path):
        print(f"[{file_path}] Waiting for output...")
        return
    try:
        with open(file_path, "r", encoding="utf-8", errors="replace") as f:
            lines = f.readlines()
            for l in lines[-num_lines:]:
                print(l.rstrip())
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

print("Dual GPU workers launched. Polling progress every 30s...")
while p0.poll() is None or p1.poll() is None:
    time.sleep(30)
    current_time = time.strftime("%H:%M:%S")
    print(f"\n==================== [GPU 0 | {current_time}] Last 15 Lines ====================")
    print_tail(log_gpu0, 15)
    print(f"==================== [GPU 1 | {current_time}] Last 15 Lines ====================")
    print_tail(log_gpu1, 15)
    sys.stdout.flush()

p0.wait()
p1.wait()
print("\nBoth GPU workers have finished execution.")


In [ ]:
import json
from pathlib import Path

import pandas as pd

results_file = Path("/kaggle/working/results.csv")
if not results_file.exists():
    raise RuntimeError("results.csv does not exist. No training runs recorded.")

df = pd.read_csv(results_file)
print("=================== COMPLETE RESULTS LEDGER (results.csv) ===================")
print(df.to_string(index=False))

summary_dict = {
    "status": "completed",
    "epochs": EPOCHS,
    "cifar10": {},
    "cifar100": {}
}

final_choice = {}

for dset in ["cifar10", "cifar100"]:
    sub_df = df[df["dataset"] == dset]
    if sub_df.empty:
        continue
    sub_df = sub_df.copy()
    sub_df["max_val_top1"] = sub_df[["best_val_top1_raw", "best_val_top1_ema"]].max(axis=1)
    best_row = sub_df.sort_values(by="max_val_top1", ascending=False).iloc[0]
    
    weight_type = "ema" if best_row["best_val_top1_ema"] >= best_row["best_val_top1_raw"] else "raw"
    run_id = f"{dset}_{best_row['optimizer']}_{best_row['lr']}"
    
    summary_dict[dset] = {
        "best_run_id": run_id,
        "optimizer": best_row["optimizer"],
        "lr": float(best_row["lr"]),
        "best_val_top1_raw": float(best_row["best_val_top1_raw"]),
        "best_val_top1_ema": float(best_row["best_val_top1_ema"]),
        "best_weight_type": weight_type,
        "best_epoch": int(best_row["best_epoch"]),
        "wall_time": float(best_row["wall_time"])
    }
    
    final_choice[dset] = {
        "run_id": run_id,
        "weight_type": weight_type
    }

print("\n=================== BEST BY VALIDATION TABLE (Never picked by test) ===================")
for dset, info in summary_dict.items():
    if dset in ["cifar10", "cifar100"]:
        print(f"[{dset.upper()}] Best Run: {info['best_run_id']} | Opt: {info['optimizer']} | LR: {info['lr']}")
        print(f"  Val Top-1 Raw: {info['best_val_top1_raw']:.2f}% | Val Top-1 EMA: {info['best_val_top1_ema']:.2f}% (Selected: {info['best_weight_type'].upper()})")
        print(f"  Best Epoch: {info['best_epoch']} | Wall Time: {info['wall_time']:.1f}s\n")

with open("/kaggle/working/summary.json", "w") as f:
    json.dump(summary_dict, f, indent=4)
print("Saved /kaggle/working/summary.json")

with open("/kaggle/working/final_choice.json", "w") as f:
    json.dump(final_choice, f, indent=4)
print("Saved /kaggle/working/final_choice.json for 04_test.ipynb evaluation.")
